In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 8 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251211_155933.csv
Loaded: NBA_DFS_20251211_155826.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Anfernee Simons,Over,11.5,-137,2025-12-12,2025-12-11T23:57:54Z,2025-12-11 15:58:26
1,Underdog,player_points,Anfernee Simons,Under,11.5,-137,2025-12-12,2025-12-11T23:57:54Z,2025-12-11 15:58:26
2,Underdog,player_points,Jaylen Brown,Over,29.5,-137,2025-12-12,2025-12-11T23:57:54Z,2025-12-11 15:58:26
3,Underdog,player_points,Jaylen Brown,Under,29.5,-137,2025-12-12,2025-12-11T23:57:54Z,2025-12-11 15:58:26
4,Underdog,player_points,Myles Turner,Over,12.5,-137,2025-12-12,2025-12-11T23:57:54Z,2025-12-11 15:58:26


In [3]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## For Post Analysis

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

singleBets = calculateSingleBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,                  
    max_player_appearances=1,  
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
singleBets.to_csv(f'notebooks/exploration/old_evs/singleBets_UD_{current_date}.csv', index=False)
singleBets

Computing predictions for 69 players...
[MIN] No data found for Ron Holland
Found 68 valid players


,NAME,LINE,SIDE,PREDICTION,MODEL_PROB,IMPLIED_PROB,EDGE,BET_EDGE,ODDS,DECIMAL_ODDS,EV,EV_PERCENT,KELLY_QUARTER,TEAM,OPPONENT
11,Jabari Smith Jr.,14.5,over,20.02,0.831,0.49,5.52,0.341,100,2.000,0.6615,66.15,0.1654,HOU,LAC
32,Maxime Raynaud,11.5,under,5.31,0.830,0.49,6.19,0.340,-101,1.990,0.6516,65.16,0.1645,SAC,DEN
12,Amen Thompson,16.5,over,22.75,0.866,0.51,6.25,0.356,-111,1.901,0.6465,64.65,0.1794,HOU,LAC
0,Anfernee Simons,11.5,over,16.62,0.829,0.53,5.12,0.299,-118,1.847,0.5307,53.07,0.1566,BOS,MIL
18,Shaedon Sharpe,22.5,over,28.95,0.828,0.53,6.45,0.298,-121,1.826,0.5127,51.27,0.1551,POR,NOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,Precious Achiuwa,7.5,under,7.01,0.412,0.50,0.49,-0.088,-104,1.962,-0.1928,-19.28,0.0000,SAC,DEN
42,Vít Krejčí,8.5,under,7.85,0.433,0.56,0.65,-0.127,-137,1.730,-0.2515,-25.15,0.0000,ATL,DET
46,Rudy Gobert,9.5,under,8.81,0.429,0.56,0.69,-0.131,-137,1.730,-0.2575,-25.75,0.0000,MIN,GSW
39,Luke Kennard,6.5,under,6.24,0.404,0.56,0.26,-0.156,-137,1.730,-0.3019,-30.19,0.0000,ATL,DET


In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=100,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/prizepicksPairs_{current_date}.csv', index=False)
# prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/underdogPairs_{current_date}_{today}.csv', index=False)
prizepicksPairs

Computing predictions for 124 players...
[MIN] No data found for A.J. Green
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 121 valid players
Generated 6581 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,IMPLIED_PROB 1,IMPLIED_PROB 2,PARLAY_PROB,PARLAY_IMPLIED_PROB,PARLAY_EDGE,EDGE 1,EDGE 2,ODDS 1,ODDS 2,PARLAY_ODDS,PARLAY_DECIMAL,EV,EV_PERCENT,KELLY_QUARTER
3569,Jordan Poole,Justin Champagnie,12.5,9.5,over,under,20.83,1.64,0.926,0.954,0.52,0.53,0.883,0.276,0.608,8.33,7.86,-115,-118,245,3.45,2.0479,204.79,0.2090
1886,Amen Thompson,Maxime Raynaud,16.5,11.5,over,under,22.75,5.31,0.866,0.830,0.51,0.49,0.719,0.250,0.469,6.25,6.19,-111,-101,278,3.78,1.7171,171.71,0.1544
864,Anfernee Simons,Reed Sheppard,11.5,10.5,over,over,16.62,16.25,0.829,0.860,0.53,0.52,0.713,0.276,0.437,5.12,5.75,-118,-115,245,3.45,1.4582,145.82,0.1488
2971,Shaedon Sharpe,Nique Clifford,22.5,8.5,over,under,28.95,2.67,0.828,0.829,0.53,0.54,0.687,0.286,0.400,6.45,5.83,-121,-123,231,3.31,1.2724,127.24,0.1377
2064,Jabari Smith Jr.,Russell Westbrook,14.0,17.5,over,under,20.02,10.85,0.854,0.792,0.56,0.52,0.676,0.291,0.385,6.02,6.65,-137,-115,223,3.23,1.1835,118.35,0.1327
3884,Jamal Murray,Paolo Banchero,24.5,21.5,over,under,29.23,15.31,0.754,0.773,0.50,0.51,0.583,0.255,0.328,4.73,6.19,-105,-110,273,3.73,1.1741,117.41,0.1075
1743,Kawhi Leonard,Stephen Curry,22.5,24.5,over,over,27.68,29.99,0.809,0.749,0.54,0.50,0.606,0.270,0.336,5.18,5.49,-122,-106,254,3.54,1.1451,114.51,0.1127
485,Payton Pritchard,Brandon Miller,16.5,21.5,over,under,20.87,14.74,0.763,0.786,0.51,0.53,0.599,0.270,0.329,4.37,6.76,-110,-118,253,3.53,1.1146,111.46,0.1101
1494,Josh Minott,Devin Vassell,5.5,11.5,over,over,9.19,16.36,0.816,0.810,0.55,0.55,0.662,0.303,0.359,3.69,4.86,-129,-127,217,3.17,1.0970,109.70,0.1264
85,Jaylen Brown,Moses Moody,29.5,8.5,over,over,34.59,11.95,0.716,0.781,0.49,0.53,0.559,0.260,0.299,5.09,3.45,100,-118,269,3.69,1.0620,106.20,0.0987


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 69 players...
[MIN] No data found for Ron Holland
Found 68 valid players
Generated 2016 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
651,Jabari Smith Jr.,Maxime Raynaud,14.5,11.5,100,-101,20.02,5.31,0.831,0.830,over,under,0.689,298,174.40,0.1463
3,Anfernee Simons,Amen Thompson,11.5,16.5,-118,-111,16.62,22.75,0.829,0.866,over,over,0.718,251,151.90,0.1513
992,Shaedon Sharpe,Nique Clifford,22.5,8.5,-121,-123,28.95,2.67,0.828,0.829,over,under,0.687,231,127.24,0.1377
1373,Russell Westbrook,Paolo Banchero,17.5,21.5,-115,-110,10.85,15.31,0.792,0.773,under,under,0.612,257,118.34,0.1151
539,Kawhi Leonard,Jamal Murray,22.5,24.5,-122,-105,27.68,29.23,0.809,0.754,over,over,0.610,255,116.70,0.1144
271,Payton Pritchard,Stephen Curry,16.5,24.5,-110,-106,20.87,29.99,0.763,0.749,over,over,0.571,271,111.87,0.1032
498,Josh Minott,Tobias Harris,5.5,13.5,-129,-108,9.19,16.88,0.816,0.726,over,over,0.593,242,102.76,0.1062
99,Jaylen Brown,De'Anthony Melton,29.5,8.5,100,-137,34.59,11.79,0.716,0.787,over,over,0.564,246,94.97,0.0965
1304,Jonas Valančiūnas,Jalen Suggs,8.5,17.5,-110,-110,10.90,12.40,0.703,0.696,over,under,0.490,264,78.19,0.0740
587,Ivica Zubac,Saddiq Bey,14.5,16.5,-115,-113,16.83,12.11,0.695,0.688,over,under,0.478,252,68.24,0.0677


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 124 players...
[MIN] No data found for A.J. Green
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 121 valid players
Generated 6581 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
3569,Jordan Poole,Justin Champagnie,12.5,9.5,-115,-118,20.83,1.64,0.926,0.954,over,under,0.883,245,204.79,0.2090
1886,Amen Thompson,Maxime Raynaud,16.5,11.5,-111,-101,22.75,5.31,0.866,0.830,over,under,0.719,278,171.71,0.1544
864,Anfernee Simons,Reed Sheppard,11.5,10.5,-118,-115,16.62,16.25,0.829,0.860,over,over,0.713,245,145.82,0.1488
2971,Shaedon Sharpe,Nique Clifford,22.5,8.5,-121,-123,28.95,2.67,0.828,0.829,over,under,0.687,231,127.24,0.1377
2064,Jabari Smith Jr.,Russell Westbrook,14.0,17.5,-137,-115,20.02,10.85,0.854,0.792,over,under,0.676,223,118.35,0.1327
3884,Jamal Murray,Paolo Banchero,24.5,21.5,-105,-110,29.23,15.31,0.754,0.773,over,under,0.583,273,117.41,0.1075
1743,Kawhi Leonard,Stephen Curry,22.5,24.5,-122,-106,27.68,29.99,0.809,0.749,over,over,0.606,254,114.51,0.1127
485,Payton Pritchard,Brandon Miller,16.5,21.5,-110,-118,20.87,14.74,0.763,0.786,over,under,0.599,253,111.46,0.1101
1494,Josh Minott,Devin Vassell,5.5,11.5,-129,-127,9.19,16.36,0.816,0.810,over,over,0.662,217,109.70,0.1264
85,Jaylen Brown,Moses Moody,29.5,8.5,100,-118,34.59,11.95,0.716,0.781,over,over,0.559,269,106.20,0.0987


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 69 players...
[MIN] No data found for Ron Holland
Found 68 valid players
Generated 34034 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
120,Anfernee Simons,Jabari Smith Jr.,Maxime Raynaud,11.5,14.5,11.5,-118,100,-101,16.62,20.02,5.31,0.829,0.831,0.830,over,over,under,0.571,635,319.88,0.1259
16825,Amen Thompson,Shaedon Sharpe,Nique Clifford,16.5,22.5,8.5,-111,-121,-123,22.75,28.95,2.67,0.866,0.828,0.829,over,over,under,0.595,529,274.03,0.1295
13843,Kawhi Leonard,Russell Westbrook,Paolo Banchero,22.5,17.5,21.5,-122,-115,-110,27.68,10.85,15.31,0.809,0.792,0.773,over,under,under,0.495,549,221.16,0.1007
6665,Payton Pritchard,Jamal Murray,Stephen Curry,16.5,24.5,24.5,-110,-105,-106,20.87,29.23,29.99,0.763,0.754,0.749,over,over,over,0.431,624,211.93,0.0849
12965,Josh Minott,Tobias Harris,De'Anthony Melton,5.5,13.5,8.5,-129,-108,-137,9.19,16.88,11.79,0.816,0.726,0.787,over,over,over,0.467,491,175.77,0.0895


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 124 players...
[MIN] No data found for A.J. Green
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 121 valid players
Generated 212161 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
149825,Jordan Poole,Maxime Raynaud,Justin Champagnie,12.5,11.5,9.5,-115,-101,-118,20.83,5.31,1.64,0.926,0.830,0.954,over,under,under,0.733,587,403.70,0.1719
41033,Anfernee Simons,Amen Thompson,Shaedon Sharpe,11.5,16.5,22.5,-118,-111,-121,16.62,22.75,28.95,0.829,0.866,0.828,over,over,over,0.594,541,280.99,0.1298
102518,Reed Sheppard,Nique Clifford,Paolo Banchero,10.5,8.5,21.5,-115,-123,-110,16.25,2.67,15.31,0.860,0.829,0.773,over,under,under,0.551,547,256.33,0.1172
94504,Jabari Smith Jr.,Russell Westbrook,Stephen Curry,14.0,17.5,24.5,-137,-115,-106,20.02,10.85,29.99,0.854,0.792,0.749,over,under,over,0.506,529,218.45,0.1032
20528,Payton Pritchard,Kawhi Leonard,Jamal Murray,16.5,22.5,24.5,-110,-122,-105,20.87,27.68,29.23,0.763,0.809,0.754,over,over,over,0.465,578,215.58,0.0932
